# Protein identity embeddings

Besides sample embeddings (SST) and attention, OmicsFM learns a third
representation: the **protein identity embeddings** of the learned-identity
checkpoints - one trainable vector per protein, shaped only by co-occurrence and
abundance patterns across the pretraining corpus. No sequence, no annotation.

These vectors carry real functional information: in the manuscript (Fig. 5D-F),
frozen identity embeddings concatenated with ESM-C sequence embeddings predict
**mean gene essentiality** (DepMap Chronos scores) better than sequence alone, and
matched controls show the gain is genuine signal, not extra dimensions. The exp11
bundle of the experiments deposit
([10.5281/zenodo.22072026](https://doi.org/10.5281/zenodo.22072026)) reproduces that
analysis end to end.

Here we simply look at the embedding space: a UMAP of all proteins, hover to see
which protein each point is (UniProt accession + gene name), coloured by how often
the protein is detected in the corpus.

**Run top to bottom with the `omicsfm` kernel.**

In [1]:
import os
import numpy as np
import pandas as pd
import torch

from omicsfm.hub import get_checkpoint, get_dataset, omicsfm_home

os.chdir(omicsfm_home())

CKPT = str(get_checkpoint("proteomics"))            # learned-identity checkpoint
DATA = str(get_dataset("proteomics_uniprot", "train"))
print("checkpoint:", CKPT)
print("data      :", DATA)


checkpoint: C:\Projects\external_omicsFM_setup\omicsFM\model\proteomics\best_model.ckpt
data      : C:\Projects\external_omicsFM_setup\omicsFM\data\proteomics_uniprot\train.h5ad


## 1. Gene names and detection frequency

Gene symbols come from the shipped FASTA headers; detection frequency (the fraction
of corpus samples a protein is detected in) is read directly from the h5ad's sparse
index - no matrix is loaded.

In [2]:
import h5py

def fasta_genes(fasta_path):
    genes, acc = {}, None
    for line in open(fasta_path, encoding="utf8"):
        if line.startswith(">"):
            parts = line.split("|")
            acc = parts[1] if len(parts) > 2 else None
            gene = None
            for tok in line.split():
                if tok.startswith("GN="):
                    gene = tok[3:]
            if acc:
                genes[acc] = gene
    return genes

GENE = fasta_genes("fasta/human_proteome_canonical_31032022.fasta")

with h5py.File(DATA, "r") as f:
    n_samples, n_proteins = f["X"].attrs["shape"]
    det = np.zeros(n_proteins, dtype=np.int64)
    idx = f["X"]["indices"]
    for start in range(0, idx.shape[0], 20_000_000):
        det += np.bincount(idx[start:start + 20_000_000], minlength=n_proteins)

detection_freq = det / n_samples
print(f"{n_proteins} proteins, {n_samples} samples; "
      f"median detection frequency {np.median(detection_freq):.3f}")


20272 proteins, 45188 samples; median detection frequency 0.076


## 2. UMAP of the learned embedding space

`visualize_proteins` pulls the identity-embedding matrix out of the checkpoint,
reduces it with UMAP, and draws an interactive scatter - hover any point for the
protein and its gene, colour shows detection frequency (log10).

One filter matters: identity embeddings are only as good as the expression context
they were trained on. Proteins detected in a handful of samples barely move from
their random initialisation and pile up as an uninformative blob, drowning the
structure of the well-trained ones. We therefore restrict the view to proteins
detected in at least 5% of corpus samples - the part of the space the model has
actually learned.

In [3]:
import umap
import plotly.express as px
from omicsfm.api import _load_model
from omicsfm.data import ExpressionDataset

ds = ExpressionDataset(DATA, num_bins=10, detect_groups=True, max_group_size=1)

# the embedding matrix: one row per protein, row order = ds.feature_names
# (row 0 is padding, hence the +1 offset)
model, _ = _load_model(CKPT, device="cpu")
emb = model.feature_emb.weight.detach().numpy()[1:]

MIN_FREQ = 0.05
mask = detection_freq >= MIN_FREQ
print(f"{mask.sum():,} of {len(mask):,} proteins detected in "
      f">= {MIN_FREQ:.0%} of samples")

xy = umap.UMAP(n_neighbors=15, min_dist=0.05, random_state=0).fit_transform(emb[mask])
df = pd.DataFrame({
    "UMAP_1": xy[:, 0], "UMAP_2": xy[:, 1],
    "protein": np.array(ds.feature_names)[mask],
    "gene": [GENE.get(p) for p in np.array(ds.feature_names)[mask]],
    "log10_detection_freq": np.log10(detection_freq[mask]),
})
fig = px.scatter(df, x="UMAP_1", y="UMAP_2", color="log10_detection_freq",
                 hover_data=["protein", "gene"], width=1000, height=800,
                 title=f"protein identity embeddings ({mask.sum():,} proteins, "
                       f"detection >= {MIN_FREQ:.0%})")
fig.update_traces(marker=dict(size=3, opacity=0.7))
fig


11,475 of 20,272 proteins detected in >= 5% of samples


C:\Users\sander\miniconda3\envs\omicsfm\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## Takeaways

- Each protein's identity embedding is learned purely from expression context, yet
  the space is organised: functionally related proteins end up near each other,
  which is what makes the embeddings predictive of gene essentiality (manuscript
  Fig. 5D-F).
- To use them in your own models: extract the matrix as
  `model.feature_emb.weight` (or follow exp11 in the experiments deposit), freeze
  it, and feed it to any downstream predictor - alone or concatenated with ESM-C
  sequence embeddings.
- The ESM-C checkpoints replace these learned identities with projected sequence
  embeddings; use those when your proteins fall outside the training vocabulary
  (see the attention tutorial).